# Hunyuan3D 2.1 — Google Colab / Tesla T4 — Shape Generation

**Clean Colab notebook for Clash of Legends.** This notebook is rebuilt from scratch in this existing file.

- Target: one NVIDIA Tesla T4 (15 GB VRAM), shape generation only.
- Output: untextured 3D meshes in GLB and OBJ formats.
- Texturing/PBR and SDXL redesign are intentionally excluded because they exceed this T4 profile.
- Runs in Google Colab from top to bottom, one cell at a time.
- Generated files stay in the Colab session until downloaded or uploaded to the private Hugging Face Dataset.

**Important:** Tencent lists about 10 GB VRAM for shape generation. A T4 has limited headroom; this notebook uses one image at a time and conservative settings. A runtime memory guard cannot guarantee that every input will fit.

Project: Clash of Legends  
Model repository: https://huggingface.co/tencent/Hunyuan3D-2.1  
Official code: https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1

## 1. License and access

Read the current Tencent Hunyuan3D 2.1 license before downloading or using the model:
https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1/blob/main/LICENSE

The model repository may require accepting its Hugging Face access conditions. This notebook reads the token from Colab Secrets; it never prints the token.

Do not use this notebook to bypass model access controls or license restrictions.

## 2. Select the Colab runtime

In Colab:
1. Open **Runtime → Change runtime type**.
2. Select **Python 3** and **T4 GPU** (if offered).
3. Save, then run the preflight cell below.

Run cells in order. Do not use **Run all** for the first setup.

In [ ]:
# PREFLIGHT — verifies the actual Colab runtime before installing anything.
import os, sys, shutil, subprocess
from pathlib import Path

print("Python:", sys.version)
print("Python executable:", sys.executable)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("No NVIDIA GPU detected. Select a GPU runtime, reconnect, then rerun this cell.")

print("GPU:")
print(gpu.stdout.strip())
free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"Free Colab disk: {free_gb:.1f} GB")
if free_gb < 20:
    raise RuntimeError("At least 20 GB free disk space is recommended before downloading the model.")

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access CUDA in this runtime.")
print("GPU name:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GB")
print("Preflight passed.")

## 3. Connect Hugging Face Secret

In Colab, open the **Secrets** (key) panel:
- Secret name: `HF_TOKEN`
- Value: your Hugging Face access token
- Enable notebook access for this secret

The token is saved by Colab Secrets. Do not paste it into a code cell.

In [ ]:
# CHECK HUGGING FACE ACCESS — token value is never displayed.
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing. Add it in Colab Secrets and enable notebook access.")

api = HfApi(token=HF_TOKEN)
try:
    who = api.whoami()
    print("Hugging Face authentication OK:", who.get("name", "authenticated user"))
except Exception as exc:
    raise RuntimeError(f"Hugging Face token check failed: {type(exc).__name__}: {exc}")

print("Next, confirm that your account has access to tencent/Hunyuan3D-2.1 on Hugging Face.")

## 4. Install shape-only dependencies

This cell keeps Colab's preinstalled PyTorch and CUDA build. It does **not** compile the PBR rasterizer, install the texture pipeline, or reinstall Torch.

In [ ]:
# INSTALL — shape-generation dependencies only; keep Colab's existing torch/torchvision.
import subprocess, sys

packages = [
    "numpy<2.3",
    "diffusers==0.30.0",
    "transformers==4.46.3",
    "accelerate==1.1.1",
    "huggingface-hub==0.30.2",
    "safetensors==0.4.4",
    "einops==0.8.0",
    "omegaconf==2.3.0",
    "pyyaml",
    "tqdm",
    "psutil",
    "trimesh==4.4.7",
    "pymeshlab>=2022.2.post3",
    "pygltflib==1.16.3",
    "xatlas",
    "rembg==2.0.65",
    "onnxruntime",
    "timm",
    "torchdiffeq",
    "opencv-python-headless==4.10.0.84",
    "imageio",
    "scikit-image",
    "ninja",
    "pybind11",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore"] + packages,
    check=True
)
print("Shape-only Python dependencies installed.")
print("Torch was not requested for reinstall. If Colab asks for a runtime restart, restart once and continue from the next cell after the preflight check.")

## 5. Download the shape model weights

Only the Hunyuan3D shape subfolder is downloaded. PBR texture weights, DINOv2, and RealESRGAN are not downloaded.

The model files are large; keep enough free disk space. If Hugging Face reports gated/restricted access, open the model page in your browser and complete its access request/conditions first.

In [ ]:
# DOWNLOAD MODEL — shape weights only.
import os, shutil
from pathlib import Path
from huggingface_hub import snapshot_download

WEIGHTS_DIR = Path("/content/hy3d_weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

shape_dir = WEIGHTS_DIR / "tencent" / "Hunyuan3D-2.1"
snapshot_download(
    repo_id="tencent/Hunyuan3D-2.1",
    allow_patterns=["hunyuan3d-dit-v2-1/*"],
    local_dir=str(shape_dir),
    token=HF_TOKEN,
)
print("Shape model files downloaded to:", shape_dir)
for p in sorted((shape_dir / "hunyuan3d-dit-v2-1").glob("*")):
    if p.is_file():
        print(f"{p.name}: {p.stat().st_size / 2**30:.2f} GB")

## 6. Download background-removal weights

The model uses U2Net through rembg when background removal is enabled. This downloads that single helper file into the session.

In [ ]:
# DOWNLOAD U2NET — no texture or PBR model is involved.
import urllib.request
from pathlib import Path

U2NET_DIR = Path("/content/hy3d_weights/u2net")
U2NET_DIR.mkdir(parents=True, exist_ok=True)
u2net_path = U2NET_DIR / "u2net.onnx"

if not u2net_path.exists() or u2net_path.stat().st_size < 1_000_000:
    temp_path = u2net_path.with_suffix(".onnx.part")
    urllib.request.urlretrieve(
        "https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx",
        temp_path
    )
    temp_path.replace(u2net_path)

print("U2Net ready:", u2net_path, f"({u2net_path.stat().st_size / 2**20:.1f} MB)")

## 7. Upload reference images

Upload PNG, JPG, JPEG, or WEBP images. For best results use a clear single-object image with the whole object visible and a plain background.

You can upload several images; generation will process them one at a time.

In [ ]:
# UPLOAD INPUT IMAGES.
from google.colab import files
from pathlib import Path
import shutil

INPUT_DIR = Path("/content/hy3d_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
allowed = {".png", ".jpg", ".jpeg", ".webp"}
saved = []
for name in uploaded:
    src = Path(name)
    if src.suffix.lower() not in allowed:
        print("Skipped unsupported file:", name)
        continue
    dst = INPUT_DIR / src.name
    if src.resolve() != dst.resolve():
        shutil.move(str(src), str(dst))
    saved.append(dst)

print(f"Saved {len(saved)} image(s) in {INPUT_DIR}")
for p in saved:
    print("-", p.name, f"{p.stat().st_size / 2**20:.1f} MB")
if not saved:
    print("No new images uploaded. You can rerun this cell to add images.")

## 8. Load the Hunyuan3D shape pipeline

This is the only model pipeline loaded on the T4. It does not load the PBR paint model.

In [ ]:
# LOAD SHAPE PIPELINE.
import os, sys, torch, gc
from pathlib import Path

REPO_DIR = Path("/content/Hunyuan3D-2.1")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git", str(REPO_DIR)],
        check=True
    )

sys.path.insert(0, str(REPO_DIR))
os.environ["HY3DGEN_MODELS"] = str(WEIGHTS_DIR)
os.environ["U2NET_HOME"] = str(U2NET_DIR)

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

torch.cuda.empty_cache()
gc.collect()

MODEL_SUBFOLDER = "hunyuan3d-dit-v2-1"
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    str(shape_dir),
    subfolder=MODEL_SUBFOLDER,
    use_safetensors=False,
    device="cuda",
    local_files_only=True,
)
print("Hunyuan3D shape pipeline loaded.")
print("GPU allocated:", round(torch.cuda.memory_allocated() / 2**30, 2), "GB")
print("GPU reserved:", round(torch.cuda.memory_reserved() / 2**30, 2), "GB")

## 9. Generate 3D meshes

The default settings are conservative for a single T4:
- One image at a time
- 30 inference steps
- Octree resolution 256
- Maximum 40,000 mesh faces

If the GPU runs out of memory, lower octree resolution to 192, then 160. Do not launch multiple workers.

In [ ]:
# GENERATE — one image at a time; preserve all successful outputs.
import gc, json, time, traceback
from pathlib import Path
from PIL import Image
import torch
from hy3dshape import FaceReducer

OUTPUT_DIR = Path("/content/hy3d_out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS = {
    "steps": 30,
    "guidance_scale": 5.0,
    "octree_resolution": 256,
    "num_chunks": 8000,
    "seed": 1234,
    "max_faces": 40000,
    "force_rembg": False,
}

face_reducer = FaceReducer()
input_files = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)
if not input_files:
    raise RuntimeError("No reference images found. Run the upload cell first.")

results = []
for index, image_path in enumerate(input_files, start=1):
    asset_name = image_path.stem
    asset_dir = OUTPUT_DIR / asset_name
    asset_dir.mkdir(parents=True, exist_ok=True)
    glb_path = asset_dir / "white_mesh.glb"
    obj_path = asset_dir / "white_mesh.obj"

    if glb_path.exists() and glb_path.stat().st_size > 1024:
        print(f"[{index}/{len(input_files)}] Keeping existing result: {asset_name}")
        results.append(glb_path)
        continue

    print(f"[{index}/{len(input_files)}] Generating: {image_path.name}")
    start = time.time()
    try:
        image = Image.open(image_path).convert("RGBA")
        image.save(asset_dir / "input.png")
        mesh = pipe(
            image=image,
            num_inference_steps=SETTINGS["steps"],
            guidance_scale=SETTINGS["guidance_scale"],
            generator=torch.Generator(device="cuda").manual_seed(SETTINGS["seed"]),
            octree_resolution=SETTINGS["octree_resolution"],
            num_chunks=SETTINGS["num_chunks"],
            output_type="trimesh",
        )[0]
        mesh = face_reducer(mesh, max_facenum=SETTINGS["max_faces"])
        tmp_glb = asset_dir / ".white_mesh.tmp.glb"
        tmp_obj = asset_dir / ".white_mesh.tmp.obj"
        mesh.export(str(tmp_glb))
        mesh.export(str(tmp_obj), include_normals=False)
        tmp_glb.replace(glb_path)
        tmp_obj.replace(obj_path)
        results.append(glb_path)
        print(f"Saved {glb_path.name} | {len(mesh.faces):,} faces | {time.time()-start:.1f}s")
        del mesh, image
    except Exception as exc:
        print(f"FAILED: {image_path.name}: {type(exc).__name__}: {exc}")
        traceback.print_exc()
    finally:
        gc.collect()
        torch.cuda.empty_cache()

print(f"Finished. Successful meshes: {len(results)} / {len(input_files)}")

## 10. Inspect and download generated files

The output is an **untextured white mesh**. PBR texturing is not part of this T4 notebook. GLB is the preferred format for previewing/importing; OBJ is also saved when export succeeds.

In [ ]:
# VALIDATE OUTPUTS AND CREATE A ZIP.
import zipfile, struct
from pathlib import Path
from IPython.display import display, FileLink

valid = []
for p in sorted(OUTPUT_DIR.glob("*/white_mesh.glb")):
    try:
        with p.open("rb") as f:
            header = f.read(12)
        ok = (
            p.stat().st_size > 1024
            and len(header) == 12
            and header[:4] == b"glTF"
            and struct.unpack("<I", header[4:8])[0] == 2
            and struct.unpack("<I", header[8:12])[0] == p.stat().st_size
        )
        if ok:
            valid.append(p)
        else:
            print("Invalid/incomplete GLB:", p)
    except OSError as exc:
        print("Could not inspect:", p, exc)

print(f"Valid GLB files: {len(valid)}")
for p in valid:
    print("-", p.parent.name, f"{p.stat().st_size / 2**20:.1f} MB")

zip_path = Path("/content/clash_of_legends_hunyuan3d_shape_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file() and not p.name.startswith("."):
            zf.write(p, p.relative_to(OUTPUT_DIR.parent))

print("ZIP created:", zip_path, f"({zip_path.stat().st_size / 2**20:.1f} MB)")
display(FileLink(str(zip_path)))

## 11. Optional: save recovery files to the private Hugging Face Dataset

This optional cell creates or reuses the private Dataset `ZzwahwahzZ/clash-of-legends-assets` and uploads the shape outputs and original input images. It does not upload tokens or notebook secrets.

Skip this cell if you only want to download the ZIP.

In [ ]:
# OPTIONAL PRIVATE DATASET CHECKPOINT.
from huggingface_hub import HfApi
from pathlib import Path
import time

DATASET_ID = "ZzwahwahzZ/clash-of-legends-assets"
token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("HF_TOKEN is missing from Colab Secrets.")

hf_api = HfApi(token=token)
hf_api.create_repo(repo_id=DATASET_ID, repo_type="dataset", private=True, exist_ok=True)
info = hf_api.repo_info(repo_id=DATASET_ID, repo_type="dataset")
if not getattr(info, "private", False):
    raise RuntimeError("Safety stop: the target Dataset is not private. Nothing was uploaded.")

upload_count = 0
for file_path in OUTPUT_DIR.rglob("*"):
    if file_path.is_file() and not file_path.name.startswith("."):
        relative = file_path.relative_to(OUTPUT_DIR).as_posix()
        hf_api.upload_file(
            path_or_fileobj=str(file_path),
            path_in_repo=f"shape_outputs/{relative}",
            repo_id=DATASET_ID,
            repo_type="dataset",
            commit_message=f"Save Hunyuan3D shape output {file_path.name}",
        )
        upload_count += 1

print(f"Uploaded {upload_count} file(s) to the private Dataset:")
print(f"https://huggingface.co/datasets/{DATASET_ID}")

## Troubleshooting

- **No GPU detected:** select T4 GPU, reconnect the runtime, then rerun the preflight cell.
- **Hugging Face 401/403:** verify the token in Colab Secrets and confirm access to the Tencent model page.
- **CUDA out of memory:** restart the runtime, rerun setup, and lower `octree_resolution` from 256 to 192 or 160. Process only one image at a time.
- **Dependency/import error:** stop at the first traceback and share the complete error. Do not continue to generation.
- **Runtime disconnected:** Colab local files may be lost. Download the ZIP or run the optional private Dataset checkpoint before ending the session.
- **No texture/color:** expected. This notebook intentionally generates shape-only GLB/OBJ outputs on a T4.

Official references:
- https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1
- https://huggingface.co/tencent/Hunyuan3D-2.1